# Предсказание на test.csv и создание submission

Обработка test повторяет шаги из `titanic_predition.ipynb`.
Все статистики (медианы, выживаемость групп по билету) считаются **по train**, чтобы не было утечки и расхождений.

In [ ]:
import pandas as pd
import numpy as np
import joblib

DATA_DIR = r"C:\projects\titanic"
MODEL_DIR = r"C:\Users\user"   # сюда сохранились rf.joblib / logreg.joblib / columns.joblib

train = pd.read_csv(fr"{DATA_DIR}\train.csv")
test = pd.read_csv(fr"{DATA_DIR}\test.csv")

model = joblib.load(fr"{MODEL_DIR}\logreg.joblib")
columns = joblib.load(fr"{MODEL_DIR}\columns.joblib")
test.isna().sum()

PassengerId      0
Pclass           0
Name             0
Sex              0
Age             86
SibSp            0
Parch            0
Ticket           0
Fare             1
Cabin          327
Embarked         0
dtype: int64

### Title
В train после замены редких остались Mr, Miss, Mrs, Master, Dr, Rev, Rare (на этом этапе заполнялся Age), потом Dr и Rev тоже ушли в Rare.

In [ ]:
def get_title(df):
    return df["Name"].str.extract(r' ([A-Za-z]+)\.', expand=False)

# Title на этапе заполнения Age (как в train-ноутбуке)
train["Title"] = get_title(train)
counts = train["Title"].value_counts()
train["Title"] = train["Title"].replace(counts[counts < 3].index, "Rare")
known_titles = set(train["Title"])          # Mr, Miss, Mrs, Master, Dr, Rev, Rare

test["Title"] = get_title(test)
test["Title"] = test["Title"].where(test["Title"].isin(known_titles), "Rare")
test["Title"].value_counts()

Title
Mr        240
Miss       78
Mrs        72
Master     21
Rare        4
Rev         2
Dr          1
Name: count, dtype: int64

### Пропуски
Age — медиана по Title из train, Fare (1 пропуск) — медиана Fare по Pclass из train, Embarked — 'S'.

In [3]:
age_medians = train.groupby("Title")["Age"].median()
test["Age"] = test["Age"].fillna(test["Title"].map(age_medians))

fare_medians = train.groupby("Pclass")["Fare"].median()
test["Fare"] = test["Fare"].fillna(test["Pclass"].map(fare_medians))

test["Embarked"] = test["Embarked"].fillna("S")

# финальная группировка титулов
test["Title"] = test["Title"].replace({"Dr": "Rare", "Rev": "Rare"})
test.isna().sum()

PassengerId      0
Pclass           0
Name             0
Sex              0
Age              0
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          327
Embarked         0
Title            0
dtype: int64

### Новые признаки
- `TicketFreq` считается по train + test: Fare — цена всего билета, а попутчики по одному билету бывают в обеих выборках.
- `GroupSurvival` — доля выживших среди попутчиков по билету **из train** (0.5, если их нет).

In [4]:
test["FamilySize"] = test["SibSp"] + test["Parch"] + 1
test["IsAlone"] = (test["FamilySize"] == 1).astype(int)

all_tickets = pd.concat([train["Ticket"], test["Ticket"]])
test["TicketFreq"] = test["Ticket"].map(all_tickets.value_counts())

test["HasCabin"] = test["Cabin"].notna().astype(int)
test["FarePerPerson"] = test["Fare"] / test["TicketFreq"]
test["IsChild"] = (test["Age"] < 12).astype(int)

test["GroupSurvival"] = test["Ticket"].map(train.groupby("Ticket")["Survived"].mean()).fillna(0.5)
test["HasGroup "] = test["TicketFreq"] > 1   # имя с пробелом, как в train

test["Sex_Pclass"] = test["Sex"].astype(str) + "_" + test["Pclass"].astype(str)
test["Sex"] = test["Sex"].map({"male": 0, "female": 1})

### Кодирование и выравнивание колонок под модель

In [5]:
passenger_id = test["PassengerId"]
X_test = test.drop(columns=["Name", "Ticket", "Cabin", "PassengerId"])
X_test = pd.get_dummies(X_test, columns=["Sex_Pclass", "Embarked", "Title"], dtype=int)

missing = set(columns) - set(X_test.columns)
extra = set(X_test.columns) - set(columns)
print("нет в test (заполним 0):", missing)
print("лишние в test (удалим):", extra)

X_test = X_test.reindex(columns=columns, fill_value=0).astype(float)
X_test.isna().sum().sum(), X_test.shape

нет в test (заполним 0): set()
лишние в test (удалим): set()


(np.int64(0), (418, 28))

### Предсказание и submission

In [6]:
pred = model.predict(X_test).astype(int)

submission = pd.DataFrame({"PassengerId": passenger_id, "Survived": pred})
submission.to_csv(fr"{DATA_DIR}\submission.csv", index=False)

print(submission.shape)
print(submission["Survived"].value_counts(normalize=True))
submission.head()

(418, 2)
Survived
0    0.62201
1    0.37799
Name: proportion, dtype: float64


,PassengerId,Survived
0,892,0
1,893,1
2,894,0
3,895,0
4,896,1
